# Random Forest Imputation for People in Need

This notebook implements a Random Forest model to impute missing `people in need` values in the UNOCHA humanitarian data.

**Data Availability:**
* We have people in need data for **2025 only** from `hpc_hno_2025`
* Requirements/funding data spans 1999-2031
* Goal: Use 2025 country-specific people in need values to fill gaps for recent historical years (2015-2024)

**Predictor Variables:**
* Country (categorical)
* Crisis type (categorical)
* Year (numeric)
* Requirements (numeric, log-transformed)
* Funding metrics (numeric)
* Country size / population (numeric, log-transformed)

**Approach:**
1. Load 2025 people in need data and aggregate to country level
2. Join with requirements/funding data on **country AND year**
3. Use 2025 data points as training set to learn relationships
4. Train Random Forest model to predict people in need based on requirements, country characteristics, and year
5. Generate predictions for missing years from 2015-2024 only

In [0]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [0]:
# Load people in need data from hpc_hno_2025
df_pin = spark.table("workspace.unochadatasets.hpc_hno_2025").toPandas()

print(f"People in need data shape: {df_pin.shape}")
print(f"Columns: {df_pin.columns.tolist()}")

# Clean column names (remove special characters)
df_pin.columns = df_pin.columns.str.replace('#', '').str.replace('+', '_').str.strip()

# Convert numeric columns
for col in ['Population', 'In Need', 'Targeted', 'Affected', 'Reached']:
    clean_col = col.replace('#', '').replace('+', '_').strip()
    if clean_col in df_pin.columns:
        df_pin[clean_col] = pd.to_numeric(df_pin[clean_col], errors='coerce')

# Aggregate to country level (sum people in need by country)
# NOTE: hpc_hno_2025 contains 2025 data only, so we'll add year=2025
df_pin_agg = df_pin.groupby('Country ISO3').agg({
    'In Need': 'sum',
    'Population': 'sum'
}).reset_index()
df_pin_agg.columns = ['countryCode', 'people_in_need', 'country_size']

# Add year column since this data is for 2025
df_pin_agg['year'] = 2025

print(f"\nAggregated people in need data shape: {df_pin_agg.shape}")
print(f"Non-null people_in_need values: {df_pin_agg['people_in_need'].notna().sum()}")
print(f"Year: 2025 (all rows)")
df_pin_agg.head()

In [0]:
# Load the requirements and funding data
df_req = spark.table("workspace.unochadatasets.fts_requirements_funding_global").toPandas()

print(f"Requirements data shape: {df_req.shape}")
print(f"Missing people_in_need: {df_req.columns.tolist()}")
print(f"\nSample data:")
df_req.head()

In [0]:
# Join people in need data with requirements data on BOTH country AND year
# Since df_pin_agg only has 2025 data, only year 2025 rows will get people_in_need values
df = df_req.merge(df_pin_agg, on=['countryCode', 'year'], how='left')

print(f"Total rows after join: {len(df)}")
print(f"Rows with people_in_need: {df['people_in_need'].notna().sum()}")
print(f"Rows missing people_in_need: {df['people_in_need'].isna().sum()}")
print(f"\nColumns: {df.columns.tolist()}")

# Check which years have people_in_need data
print(f"\nYears with people_in_need data:")
print(df[df['people_in_need'].notna()]['year'].value_counts().sort_index())

# Fill missing country_size with median
if 'country_size' in df.columns:
    median_size = df['country_size'].median()
    df['country_size'].fillna(median_size, inplace=True)
    print(f"\nFilled missing country_size with median: {median_size:,.0f}")

df.head(10)

In [0]:
# Load severity index data from the most recent available table
df_severity = spark.table('workspace.severityindex.predict_pin_202602').toPandas()

print(f"Severity data shape: {df_severity.shape}")
print(f"Unique countries: {df_severity['iso3'].nunique()}")

# Convert numeric columns
numeric_severity_cols = ['inform_severity_index', 'impact_of_the_crisis', 'geographical', 
                         'human', 'conditions_of_people_affected', 'concentration_of_conditions',
                         'complexity_of_the_crisis', 'society_and_safety', 'operating_environment']

for col in numeric_severity_cols:
    if col in df_severity.columns:
        df_severity[col] = pd.to_numeric(df_severity[col], errors='coerce')

# Select key severity features to join
severity_features = df_severity[['iso3', 'inform_severity_index', 'inform_severity_category',
                                  'impact_of_the_crisis', 'complexity_of_the_crisis', 
                                  'society_and_safety', 'human', 'conditions_of_people_affected']].copy()

# Join severity data with the main dataframe on country code
df = df.merge(severity_features, left_on='countryCode', right_on='iso3', how='left')

print(f"\nRows with severity data: {df['inform_severity_index'].notna().sum()}")
print(f"Rows missing severity data: {df['inform_severity_index'].isna().sum()}")

# Fill missing severity values with median
for col in ['inform_severity_index', 'impact_of_the_crisis', 'complexity_of_the_crisis', 
            'society_and_safety', 'human', 'conditions_of_people_affected']:
    if col in df.columns:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} missing values with median: {median_val:.2f}")

print(f"\nUpdated dataframe shape: {df.shape}")
df[['countryCode', 'year', 'requirements', 'people_in_need', 'inform_severity_index', 
    'impact_of_the_crisis', 'complexity_of_the_crisis']].head(10)

In [0]:
# Separate complete cases (for training) from missing cases (for imputation)
df_complete = df[df['people_in_need'].notna()].copy()
df_missing = df[df['people_in_need'].isna()].copy()

print(f"Complete cases (training set): {len(df_complete)}")
print(f"Missing cases (to impute): {len(df_missing)}")

# Remove rows with non-positive values in key columns
df_complete = df_complete[
    (df_complete['people_in_need'] > 0) & 
    (df_complete['requirements'].notna()) & 
    (df_complete['requirements'] > 0)
].copy()

print(f"Complete cases after removing non-positive values: {len(df_complete)}")

# Fill any remaining missing values
for col in ['requirements', 'funding', 'percentFunded', 'country_size']:
    if col in df_complete.columns:
        median_val = df_complete[col].median()
        df_complete[col].fillna(median_val, inplace=True)
        df_missing[col].fillna(median_val, inplace=True)
        missing_count = df_complete[col].isna().sum()
        if missing_count == 0:
            print(f"✓ {col}: all filled")
        else:
            print(f"⚠ {col}: still {missing_count} missing")

In [0]:
# Encode categorical variables
le_country = LabelEncoder()
le_crisis = LabelEncoder()

# Fit encoders on complete cases
df_complete['country_encoded'] = le_country.fit_transform(df_complete['countryCode'])
df_complete['crisis_encoded'] = le_crisis.fit_transform(df_complete['typeName'])

# Create log-transformed features
df_complete['log_requirements'] = np.log1p(df_complete['requirements'])
df_complete['log_country_size'] = np.log1p(df_complete['country_size'])
df_complete['log_funding'] = np.log1p(df_complete['funding'])
df_complete['log_pin'] = np.log1p(df_complete['people_in_need'])

# Prepare features and target - NOW INCLUDING SEVERITY INDICES
feature_cols = ['country_encoded', 'crisis_encoded', 'year', 'log_requirements', 'log_country_size', 
                'log_funding', 'percentFunded', 'inform_severity_index', 'impact_of_the_crisis', 
                'complexity_of_the_crisis', 'society_and_safety', 'human', 'conditions_of_people_affected']
X_train = df_complete[feature_cols]
y_train = df_complete['log_pin']

print(f"Training features shape: {X_train.shape}")
print(f"Target variable shape: {y_train.shape}")
print(f"\nFeature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")
print(f"\nFeature summary:")
print(X_train.describe())

In [0]:
# Split data for validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Train Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100, 
    max_depth=10, 
    random_state=42, 
    n_jobs=-1
)

print("Training Random Forest model...")
rf_model.fit(X_train_split, y_train_split)

# Evaluate on validation set
train_score = rf_model.score(X_train_split, y_train_split)
val_score = rf_model.score(X_val, y_val)

print(f"\n✓ Model trained successfully!")
print(f"Training R²: {train_score:.4f}")
print(f"Validation R²: {val_score:.4f}")
print(f"Overfit gap: {train_score - val_score:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance:")
print(feature_importance)

In [0]:
# Filter missing data to only include years from 2015 to 2025
df_missing = df_missing[(df_missing['year'] >= 2015) & (df_missing['year'] <= 2025)].copy()

print(f"Missing cases to impute (2015-2025): {len(df_missing)}")
print(f"Year range: {df_missing['year'].min()}-{df_missing['year'].max()}")

# Prepare missing data for prediction
if len(df_missing) > 0:
    # Encode categorical variables for missing data
    # Handle unseen categories by using the most common category
    df_missing['country_encoded'] = df_missing['countryCode'].map(
        lambda x: le_country.transform([x])[0] if x in le_country.classes_ else le_country.transform([le_country.classes_[0]])[0]
    )
    df_missing['crisis_encoded'] = df_missing['typeName'].map(
        lambda x: le_crisis.transform([x])[0] if x in le_crisis.classes_ else le_crisis.transform([le_crisis.classes_[0]])[0]
    )
    
    # Create log-transformed features
    df_missing['log_requirements'] = np.log1p(df_missing['requirements'])
    df_missing['log_country_size'] = np.log1p(df_missing['country_size'])
    df_missing['log_funding'] = np.log1p(df_missing['funding'])
    
    # Prepare features
    X_missing = df_missing[feature_cols]
    
    # Generate predictions
    print("\nGenerating predictions for years 2015-2025...")
    log_predictions = rf_model.predict(X_missing)
    predictions = np.expm1(log_predictions)  # Reverse log transformation
    
    # Add predictions to dataframe
    df_missing['people_in_need_predicted'] = predictions
    
    print(f"\n✓ Generated {len(predictions)} predictions for years {df_missing['year'].min()}-{df_missing['year'].max()}")
    print(f"\nPrediction statistics:")
    print(f"  Min: {predictions.min():,.0f}")
    print(f"  Max: {predictions.max():,.0f}")
    print(f"  Mean: {predictions.mean():,.0f}")
    print(f"  Median: {np.median(predictions):,.0f}")
    
    # Display sample predictions
    print(f"\nSample predictions:")
    display(df_missing[['countryCode', 'name', 'year', 'requirements', 'people_in_need_predicted']].head(10))
else:
    print("⚠ No missing values to predict!")

In [0]:
# Count unique countries with predictions
unique_countries = df_missing['countryCode'].nunique()
countries_list = sorted(df_missing['countryCode'].unique())

print(f"Number of unique countries with predictions: {unique_countries}")
print(f"\nCountries: {', '.join(countries_list)}")

# Show predictions breakdown by country
print(f"\nPredictions per country:")
country_counts = df_missing.groupby('countryCode').size().sort_values(ascending=False)
display(country_counts)

## Create Complete HNO Dataset with Predictions

This section creates a new table combining:
* **Actual 2025 data** from `hpc_hno_2025`
* **Predicted values for 2015-2024** based on the Random Forest model

The predictions are at country-year level, so we'll:
1. Replicate the HNO structure for each historical year
2. Proportionally distribute predicted totals across sectors/categories
3. Save to a new table: `workspace.unochadatasets.hpc_hno_complete_with_predictions`

In [0]:
# Get the country-level predictions from earlier
print(f"Predictions available: {len(df_missing)} records for years {df_missing['year'].min()}-{df_missing['year'].max()}")
print(f"\nCountries with predictions: {df_missing['countryCode'].nunique()}")

# Load original 2025 HNO data to use as template
df_hno_2025 = spark.table('workspace.unochadatasets.hpc_hno_2025').toPandas()

# Clean column names
df_hno_2025.columns = df_hno_2025.columns.str.replace('#', '').str.replace('+', '_').str.strip()

print(f"\nOriginal HNO 2025 data: {df_hno_2025.shape}")
print(f"Columns: {df_hno_2025.columns.tolist()}")

# Add year column to 2025 data
df_hno_2025['year'] = 2025

# Convert numeric columns
for col in ['Population', 'In Need', 'Targeted', 'Affected', 'Reached']:
    clean_col = col.replace('#', '').replace('+', '_').strip()
    if clean_col in df_hno_2025.columns:
        df_hno_2025[clean_col] = pd.to_numeric(df_hno_2025[clean_col], errors='coerce')

# Get country-level totals for 2025 (to calculate proportions)
df_country_totals_2025 = df_hno_2025.groupby('Country ISO3').agg({
    'In Need': 'sum',
    'Population': 'sum'
}).reset_index()
df_country_totals_2025.columns = ['countryCode', 'total_in_need_2025', 'total_population_2025']

print(f"\nCountry totals for 2025: {len(df_country_totals_2025)} countries")
df_country_totals_2025.head()

In [0]:
# Create a template from 2025 data (one country's structure)
# We'll replicate this structure for each year 2015-2024

historical_data_list = []

# Get predictions grouped by country and year
predictions_grouped = df_missing.groupby(['countryCode', 'year'])['people_in_need_predicted'].sum().reset_index()

print(f"Creating historical data for {len(predictions_grouped)} country-year combinations...")
print(f"Years: {predictions_grouped['year'].unique()}")

# For each country-year with predictions
for idx, row in predictions_grouped.iterrows():
    country = row['countryCode']
    year = int(row['year'])
    predicted_total = row['people_in_need_predicted']
    
    # Get the 2025 structure for this country
    country_template = df_hno_2025[df_hno_2025['Country ISO3'] == country].copy()
    
    if len(country_template) == 0:
        # Country not in 2025 data, skip
        continue
    
    # Get 2025 total for proportional scaling
    country_2025_total = df_country_totals_2025[df_country_totals_2025['countryCode'] == country]['total_in_need_2025'].values
    
    if len(country_2025_total) > 0 and country_2025_total[0] > 0:
        scaling_factor = predicted_total / country_2025_total[0]
    else:
        # If no 2025 data, distribute equally across template rows
        scaling_factor = predicted_total / len(country_template)
    
    # Scale the In Need values proportionally
    country_template['In Need'] = country_template['In Need'] * scaling_factor
    country_template['year'] = year
    
    historical_data_list.append(country_template)
    
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(predictions_grouped)} country-year combinations...")

# Combine all historical data
if historical_data_list:
    df_hno_historical = pd.concat(historical_data_list, ignore_index=True)
    print(f"\n✓ Created historical data: {df_hno_historical.shape}")
    print(f"  Years: {sorted(df_hno_historical['year'].unique())}")
    print(f"  Countries: {df_hno_historical['Country ISO3'].nunique()}")
    print(f"\nSample historical data:")
    display(df_hno_historical[['Country ISO3', 'year', 'Description', 'Cluster', 'Category', 'In Need']].head(10))
else:
    print("⚠ No historical data created!")
    df_hno_historical = pd.DataFrame()

In [0]:
# Combine 2025 actual data with historical predictions
df_hno_complete = pd.concat([df_hno_2025, df_hno_historical], ignore_index=True)

print(f"Complete dataset shape: {df_hno_complete.shape}")
print(f"Years covered: {sorted(df_hno_complete['year'].unique())}")
print(f"Countries: {df_hno_complete['Country ISO3'].nunique()}")
print(f"\nRecords per year:")
print(df_hno_complete['year'].value_counts().sort_index())

# Clean column names for Delta Lake (remove spaces and special characters)
df_hno_complete.columns = (
    df_hno_complete.columns
    .str.replace(' ', '_')
    .str.replace('#', '')
    .str.replace('+', '_')
    .str.replace('(', '')
    .str.replace(')', '')
    .str.strip()
)

print(f"\nCleaned column names: {df_hno_complete.columns.tolist()}")

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df_hno_complete)

# Save to new table
table_name = 'workspace.unochadatasets.hpc_hno_complete_with_predictions'

print(f"\nSaving to table: {table_name}")
spark_df.write.mode('overwrite').saveAsTable(table_name)

print(f"\n✓ Table created successfully!")
print(f"  Table: {table_name}")
print(f"  Total records: {df_hno_complete.shape[0]:,}")
print(f"  Years: {df_hno_complete['year'].min()}-{df_hno_complete['year'].max()}")
print(f"  Countries: {df_hno_complete['Country_ISO3'].nunique()}")

In [0]:
# Verify the new table
df_verify = spark.table('workspace.unochadatasets.hpc_hno_complete_with_predictions').toPandas()

print(f"Verification of new table:")
print(f"  Total records: {len(df_verify):,}")
print(f"  Years: {sorted(df_verify['year'].unique())}")
print(f"  Countries: {df_verify['Country_ISO3'].nunique()}")

# Show summary by year
print(f"\nRecords per year:")
year_summary = df_verify.groupby('year').agg({
    'Country_ISO3': 'nunique',
    'In_Need': ['sum', 'count']
}).round(0)
year_summary.columns = ['Countries', 'Total_In_Need', 'Record_Count']
print(year_summary.sort_index())

print(f"\nSample records from different years:")
for year in [2015, 2020, 2025]:
    if year in df_verify['year'].values:
        sample = df_verify[df_verify['year'] == year][['Country_ISO3', 'year', 'Description', 'In_Need']].head(2)
        print(f"\n{year}:")
        display(sample)